## Setup - import and connection

In [1]:
import pandas as pd
from sqlalchemy import create_engine
import os
from dotenv import load_dotenv


load_dotenv()

user = os.getenv('DB_USER')
password = os.getenv('DB_PASSWORD')
host = os.getenv('DB_HOST')
port = os.getenv('DB_PORT')
dbname = os.getenv('DB_NAME')

engine = create_engine(f"postgresql://{user}:{password}@{host}:{port}/{dbname}")

## Load data

In [2]:
fact_brent_prices = pd.read_sql("SELECT * FROM fact_brent_prices", engine)
fact_eur_usd_rates = pd.read_sql("SELECT * FROM fact_eur_usd_rates", engine)
fact_pump_prices_pretax = pd.read_sql("SELECT * FROM fact_pump_prices_pretax", engine)

In [3]:
fact_brent_prices

,obs_date,close_price
0,2007-07-30,75.739998
1,2007-07-31,77.050003
2,2007-08-01,75.349998
3,2007-08-02,75.760002
4,2007-08-03,74.750000
...,...,...
4732,2026-08-05,79.449997
4733,2026-08-06,82.489998
4734,2026-08-07,83.550003
4735,2026-08-10,87.720001


In [4]:
fact_pump_prices_pretax.shape

(1078, 3)

In [5]:
fact_pump_prices_pretax = fact_pump_prices_pretax.sort_values('week_date')
fact_brent_prices = fact_brent_prices.sort_values('obs_date')

df = pd.merge_asof(
    fact_pump_prices_pretax,
    fact_brent_prices,
    left_on='week_date',
    right_on='obs_date',
    direction='backward',
)
df.shape

(1078, 5)

In [6]:
df = df.rename(columns={'obs_date': 'brent_date'})
df.head()

,week_date,petrol_price_pretax,diesel_price_pretax,brent_date,close_price
0,2005-01-03,391.20,437.73,NaT,NaN
1,2005-01-10,391.20,437.73,NaT,NaN
2,2005-01-17,335.01,416.24,NaT,NaN
3,2005-01-24,335.01,416.24,NaT,NaN
4,2005-01-31,335.01,416.24,NaT,NaN


In [7]:
fact_eur_usd_rates

,obs_date,close_rate
0,2003-12-01,1.196501
1,2003-12-02,1.208897
2,2003-12-03,1.212298
3,2003-12-04,1.208094
4,2003-12-05,1.218695
...,...,...
5883,2026-08-05,1.153243
5884,2026-08-06,1.155735
5885,2026-08-07,1.152472
5886,2026-08-10,1.155642


In [8]:
fact_eur_usd_rates = fact_eur_usd_rates.sort_values('obs_date')

df = pd.merge_asof(
    df,
    fact_eur_usd_rates,
    left_on='week_date',
    right_on='obs_date',
    direction='backward',
)

df.shape

(1078, 7)

In [22]:
df = df.rename(columns={'obs_date': 'eur_usd_date'})
df.tail()

,week_date,petrol_price_pretax,diesel_price_pretax,brent_date,close_price,eur_usd_date,close_rate
1073,2026-07-06,802.161463,919.289024,2026-07-06,71.989998,2026-07-06,1.143772
1074,2026-07-13,788.096423,900.264634,2026-07-13,83.300003,2026-07-13,1.140446
1075,2026-07-20,794.356585,904.898780,2026-07-20,89.220001,2026-07-20,1.142766
1076,2026-07-27,824.844390,950.996341,2026-07-27,88.360001,2026-07-27,1.139497
1077,2026-08-03,873.299675,1027.337805,2026-08-03,83.769997,2026-08-03,1.154401


In [10]:
df_clean = df.dropna(subset=['close_price']).reset_index(drop=True)
df_clean['brent_eur'] = df_clean['close_price'] / df_clean['close_rate']
df_clean.shape



(952, 8)

In [21]:
df_clean.tail()

,week_date,petrol_price_pretax,diesel_price_pretax,brent_date,close_price,eur_usd_date,close_rate,brent_eur,brent_pct,pump_pct,eur_usd_pct
947,2026-07-06,802.161463,919.289024,2026-07-06,71.989998,2026-07-06,1.143772,62.940857,-0.015858,-0.031794,0.004575
948,2026-07-13,788.096423,900.264634,2026-07-13,83.300003,2026-07-13,1.140446,73.041606,0.157105,-0.017534,-0.002908
949,2026-07-20,794.356585,904.898780,2026-07-20,89.220001,2026-07-20,1.142766,78.073744,0.071068,0.007943,0.002034
950,2026-07-27,824.844390,950.996341,2026-07-27,88.360001,2026-07-27,1.139497,77.542968,-0.009639,0.038381,-0.002860
951,2026-08-03,873.299675,1027.337805,2026-08-03,83.769997,2026-08-03,1.154401,72.565758,-0.051947,0.058745,0.013079


In [12]:
df_clean.to_parquet('../data/processed/weekly_brent_eur_pump.parquet', index=False)

## Weekly % Change

Switching from log-differences to simple percent change — same idea
(week-over-week movement), but reads directly as "% change", no log
explanation needed for a business audience.

In [13]:
df_clean['brent_pct'] = df_clean['close_price'].pct_change()
df_clean['pump_pct'] = df_clean['petrol_price_pretax'].pct_change()
df_clean['eur_usd_pct'] = df_clean['close_rate'].pct_change()

In [14]:
df_clean.head()

,week_date,petrol_price_pretax,diesel_price_pretax,brent_date,close_price,eur_usd_date,close_rate,brent_eur,brent_pct,pump_pct,eur_usd_pct
0,2007-07-30,528.39,545.17,2007-07-30,75.739998,2007-07-30,1.371197,55.236424,NaN,NaN,NaN
1,2007-08-06,528.39,545.16,2007-08-06,71.169998,2007-08-06,1.379691,51.584013,-0.060338,0.0,0.006195
2,2007-08-13,528.39,545.16,2007-08-13,70.230003,2007-08-13,1.361304,51.590259,-0.013208,0.0,-0.013327
3,2007-08-20,528.39,545.16,2007-08-20,69.849998,2007-08-20,1.347491,51.837083,-0.005411,0.0,-0.010147
4,2007-08-27,528.39,545.16,2007-08-27,70.949997,2007-08-27,1.363791,52.024085,0.015748,0.0,0.012097


In [15]:
(df_clean['pump_pct'] == 0).sum() / len(df_clean)

np.float64(0.38340336134453784)

In [16]:
fact_pump_prices_pretax['petrol_price_pretax'].diff().eq(0).sum() / len(fact_pump_prices_pretax)

np.float64(0.42857142857142855)

In [17]:
df_clean['week_date'].dt.year[df_clean['pump_pct'] == 0].value_counts().sort_index()

week_date
2007    16
2008    38
2009    33
2010    38
2011    38
2012    38
2013    39
2014    34
2015    23
2016    20
2017    16
2018    18
2019     5
2020     2
2021     2
2022     5
Name: count, dtype: int64

In [18]:
fact_pump_prices_pretax['week_date'].max()

Timestamp('2026-08-03 00:00:00')

In [19]:
fact_brent_prices['obs_date'].max()

Timestamp('2026-08-11 00:00:00')

In [20]:
df['week_date'].max()

Timestamp('2026-08-03 00:00:00')

In [23]:
print(df_clean['week_date'].dt.year[df_clean['pump_pct'] == 0].value_counts().sort_index().to_string())

week_date
2007    16
2008    38
2009    33
2010    38
2011    38
2012    38
2013    39
2014    34
2015    23
2016    20
2017    16
2018    18
2019     5
2020     2
2021     2
2022     5


In [24]:
df_clean['week_date'].max()

Timestamp('2026-08-03 00:00:00')

In [25]:
print(df_clean['week_date'].dt.year[df_clean['pump_pct'] == 0].value_counts().sort_index().to_string())

week_date
2007    16
2008    38
2009    33
2010    38
2011    38
2012    38
2013    39
2014    34
2015    23
2016    20
2017    16
2018    18
2019     5
2020     2
2021     2
2022     5


In [26]:
df_model = df_clean[df_clean['week_date'] >= '2019-01-01'].reset_index(drop=True)
df_model.shape

(387, 11)

In [27]:
df_model['brent_pct_lag1'] = df_model['brent_pct'].shift(1)
df_model['brent_pct_lag2'] = df_model['brent_pct'].shift(2)

df_model['eur_usd_pct_lag1'] = df_model['eur_usd_pct'].shift(1)
df_model['eur_usd_pct_lag2'] = df_model['eur_usd_pct'].shift(2)

df_reg = df_model.dropna(subset=[
    'pump_pct', 'brent_pct', 'brent_pct_lag1', 'brent_pct_lag2',
    'eur_usd_pct', 'eur_usd_pct_lag1', 'eur_usd_pct_lag2'
])

df_reg.shape

(385, 15)

In [28]:
import statsmodels.api as sm

X = df_reg[[
    'brent_pct', 'brent_pct_lag1', 'brent_pct_lag2',
    'eur_usd_pct', 'eur_usd_pct_lag1', 'eur_usd_pct_lag2'
]]
X = sm.add_constant(X)

y = df_reg['pump_pct']

model = sm.OLS(y, X).fit()
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:               pump_pct   R-squared:                       0.210
Model:                            OLS   Adj. R-squared:                  0.197
Method:                 Least Squares   F-statistic:                     16.72
Date:                Wed, 12 Aug 2026   Prob (F-statistic):           4.01e-17
Time:                        16:12:57   Log-Likelihood:                 897.74
No. Observations:                 385   AIC:                            -1781.
Df Residuals:                     378   BIC:                            -1754.
Df Model:                           6                                         
Covariance Type:            nonrobust                                         
                       coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------------
const                0.0010      0.001  

## Interpretation

Brent's effect on Irish pump prices is delayed, not immediate — the same-week
coefficient isn't significant (p=0.132), but it becomes significant and grows
stronger over the following two weeks:

- 1 week later: 10% Brent increase → ~1.1% pump price increase (p<0.001)
- 2 weeks later: 10% Brent increase → ~1.5% pump price increase (p<0.001)

This gives a ~1-2 week window between an oil price move and its full effect
at the pump — actionable lead time for treasury/procurement teams planning
hedges or budget adjustments.

EUR/USD's effect is weaker and less consistent — only the 2-week lag is
significant, and with an unexpected negative sign, likely noise rather than
a real relationship. Not a reliable driver on its own.

Model explains ~21% of weekly pump price variation (R²=0.21) — expected,
since retail pricing also reflects competition, margins, and local factors
not captured here.

In [30]:
df_reg.to_parquet('../data/processed/weekly_pump_brent_eur_2019_2026.parquet', index=False)